In [ ]:
# Install all libraries with pinned versions that work together with Phi-3
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q
!pip install transformers==4.44.2 \
             datasets==2.21.0 \
             trl==0.8.6 \
             peft==0.12.0 \
             accelerate==0.33.0 \
             bitsandbytes>=0.46.1 \
             pandas \
             scikit-learn \
             kagglehub -q

# Restart runtime after install to avoid numpy binary conflict
import os
os.kill(os.getpid(), 9)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.
pytensor 2.37.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9",

In [ ]:
import bitsandbytes as bnb, transformers, peft, accelerate
print("bitsandbytes:", bnb.__version__)
print("transformers:", transformers.__version__)
print("peft        :", peft.__version__)
print("accelerate  :", accelerate.__version__)

bitsandbytes: 0.49.2
transformers: 4.44.2
peft        : 0.12.0
accelerate  : 0.33.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch

cuda_available = torch.cuda.is_available()
print(f"CUDA available : {cuda_available}")

if cuda_available:
    print(f" GPU  : {torch.cuda.get_device_name(0)}")
    print(f" VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")
else:
    print(" No GPU found. Please ensure your Colab runtime is set to GPU.")

Mounted at /content/drive
CUDA available : True
 GPU  : NVIDIA A100-SXM4-80GB
 VRAM : 85.1GB


##LOAD UCI CREDIT DATASET

In [ ]:
import pandas as pd
import requests
from io import BytesIO

# URL to the dataset file (usually found under 'Data Folder' on the UCI page)
data_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00350/default%20of%20credit%20card%20clients.xls"

# Download the file
response = requests.get(data_url)
response.raise_for_status() # Raise an exception for bad status codes

# Read the Excel file into a pandas DataFrame
df = pd.read_excel(BytesIO(response.content), header=1) # header=1 to skip the first row that contains column type descriptions

# The dataset has a column name that starts with 'ID', which seems to be the ID column.
# The target variable `default payment next month` is the correct column name.

print(f" Dataset loaded: {df.shape}")
print(f"Defaults: {len(df[df['default payment next month']==1])} / {len(df)}")
df.head()

 Dataset loaded: (30000, 25)
Defaults: 6636 / 30000


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [ ]:
from datasets import Dataset

# Safety check: rename column if not already renamed
if 'default payment next month' in df.columns:
    df.rename(columns={"default payment next month": "label"}, inplace=True)
    print(" Column renamed to 'label'")

# ── Define the function FIRST
def format_phi3(row):
    features = (
        f"Age: {int(row['AGE'])}, "
        f"Sex: {int(row['SEX'])}, "
        f"Marriage: {int(row['MARRIAGE'])}, "
        f"PAY_0: {int(row['PAY_0'])}, "
        f"Limit: ${int(row['LIMIT_BAL']):,}"
    )
    label = "Default" if int(row['label']) == 1 else "No default"
    return (
        f"<|user|>\n"
        f"You are a credit risk model. "
        f"Answer only with 'Default' or 'No default'.\n\n"
        f"Client:\n{features}\n"
        f"<|assistant|>\n"
        f"{label}<|end|>"
    )

# ── THEN use it
df_small = df.sample(n=5000, random_state=42)
df_small['text'] = df_small.apply(format_phi3, axis=1)
dataset = Dataset.from_pandas(df_small[['text']])

print(f" Total dataset      : {len(df)} rows")
print(f" Training samples   : {len(dataset)}")
print(f" Defaults in sample : {df_small['label'].sum()}")
print(f"\nSample prompt:")
print(dataset[0]['text'])

 Column renamed to 'label'
 Total dataset      : 30000 rows
 Training samples   : 5000
 Defaults in sample : 1113

Sample prompt:
<|user|>
You are a credit risk model. Answer only with 'Default' or 'No default'.

Client:
Age: 25, Sex: 1, Marriage: 2, PAY_0: 0, Limit: $30,000
<|assistant|>
No default<|end|>


In [ ]:
from datasets import Dataset

# Safety check: rename column if not already renamed
if 'default payment next month' in df.columns:
    df.rename(columns={"default payment next month": "label"}, inplace=True)
    print(" Column renamed to 'label'")

# ── Use 5,000 samples instead of 30,000
df_small = df.sample(n=5000, random_state=42)  # random_state=42 means
                                                # same 5000 every run
df_small['text'] = df_small.apply(format_phi3, axis=1)
dataset = Dataset.from_pandas(df_small[['text']])

print(f" Total dataset : {len(df)} rows")
print(f" Using         : {len(dataset)} samples for training")
print(f" Defaults in sample : {df_small['label'].sum()}")
print(f"\nSample prompt:")
print(dataset[0]['text'])

 Total dataset : 30000 rows
 Using         : 5000 samples for training
 Defaults in sample : 1113

Sample prompt:
<|user|>
You are a credit risk model. Answer only with 'Default' or 'No default'.

Client:
Age: 25, Sex: 1, Marriage: 2, PAY_0: 0, Limit: $30,000
<|assistant|>
No default<|end|>


##Format for phi-3

In [ ]:
def format_phi3(row):
    features = (
        f"Age: {int(row['AGE'])}, "
        f"Sex: {int(row['SEX'])}, "
        f"Marriage: {int(row['MARRIAGE'])}, "
        f"PAY_0: {int(row['PAY_0'])}, "
        f"Limit: ${int(row['LIMIT_BAL']):,}"
    )
    label = "Default" if int(row['label']) == 1 else "No default"
    return (
        f"<|user|>\n"
        f"You are a credit risk model. "
        f"Answer only with 'Default' or 'No default'.\n\n"
        f"Client:\n{features}\n"
        f"<|assistant|>\n"
        f"{label}<|end|>"
    )

df['text'] = df.apply(format_phi3, axis=1)
dataset = Dataset.from_pandas(df[['text']])

print(f"\n Formatted: {len(dataset)} samples")
print("\nSample 1 (No default):")
print(dataset[0]['text'])
print("\nSample 2 (Default case):")
print(format_phi3(df[df['label'] == 1].iloc[0]))


 Formatted: 30000 samples

Sample 1 (No default):
<|user|>
You are a credit risk model. Answer only with 'Default' or 'No default'.

Client:
Age: 24, Sex: 2, Marriage: 1, PAY_0: 2, Limit: $20,000
<|assistant|>
Default<|end|>

Sample 2 (Default case):
<|user|>
You are a credit risk model. Answer only with 'Default' or 'No default'.

Client:
Age: 24, Sex: 2, Marriage: 1, PAY_0: 2, Limit: $20,000
<|assistant|>
Default<|end|>


##Load Phi3 mini + QLoRA

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Loading Phi-3 base model...")
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True
)

tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
print(" Model ready for training")

Loading Phi-3 base model...


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_phi3.py: 0.00B [00:00, ?B/s]

...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

trainable params: 3,145,728 || all params: 3,824,225,280 || trainable%: 0.0823
 Model ready for training


In [ ]:
# Re-sample back to 5,000 to fix the overwrite from Cell 11
df_small = df.sample(n=5000, random_state=42)
df_small['text'] = df_small.apply(format_phi3, axis=1)
dataset = Dataset.from_pandas(df_small[['text']])
print(f" Dataset fixed: {len(dataset)} samples ready for training")

 Dataset fixed: 5000 samples ready for training


##START TRAINING

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

tokenizer.padding_side = 'right'

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/phi3-credit-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    warmup_steps=20,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    optim="paged_adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    dataloader_num_workers=0,
    dataloader_drop_last=True,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args,
    packing=False,
)

print("TRAINING STARTING...")
print(f"Total samples : {len(dataset)}")
print(f"Epochs        : 3")
print(f"Estimated time: ~25-35 minutes")
trainer.train()
print(" TRAINING COMPLETE!")

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

TRAINING STARTING...
Total samples : 5000
Epochs        : 3
Estimated time: ~25-35 minutes


Step,Training Loss
10,2.999200
20,2.142200
30,0.793400
40,0.217700
50,0.177300
60,0.171800
70,0.170400
80,0.169100
90,0.162600
100,0.169800


 TRAINING COMPLETE!


##Save the model

In [ ]:
save_dir = "/content/drive/MyDrive/phi3-credit-finetuned-final"
trainer.model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print("Saved to:", save_dir)


##reloading model for inference

In [ ]:
import warnings
warnings.filterwarnings('ignore')

!pip install -U bitsandbytes>=0.46.1 -q

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

save_dir = "/content/drive/MyDrive/phi3-credit-finetuned-final"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = PeftModel.from_pretrained(base_model, save_dir)
model.eval()


In [ ]:
!pip install bitsandbytes -q

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from peft import PeftModel
import torch

# --- Code copied from cell E9Wu3Y6FzEcw to ensure model and tokenizer are defined ---
save_dir = "/content/drive/MyDrive/phi3-credit-finetuned-final"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = PeftModel.from_pretrained(base_model, save_dir)
model.eval()


gen = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=5,
    do_sample=False,
    temperature=0.0,
)

prompt = """<|user|>
Predict credit card default risk:
Age: 45, Sex: 1, Marriage: 1, PAY_0: 2, Limit: $50,000
<|assistant|>
"""

out = gen(prompt)[0]["generated_text"]
print(out)
